In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l7.4 QLoRA

> 🎯 **Goal:** Combine LoRA with a compressed (4-bit) frozen base so even large
> models fit on a single small GPU.

**Why this matters.** LoRA shrinks the *trainable* parameters, but the frozen base
still has to sit in memory in full precision, and for a large model that base alone
can overflow a consumer GPU. QLoRA closes that gap and is the technique that
genuinely democratized fine-tuning: it is why hobbyists can adapt multi-billion
parameter models on a single rented or desktop GPU.

**The intuition.** Take the LoRA idea (bolt small adjustment knobs onto a frozen
model) and first *compress the frozen model itself* so it takes up far less room.
You store the base in a coarse, low-resolution form, then turn your full-precision
knobs on top. The base is fuzzier, but the sharp little adapter learns around the
fuzziness and recovers the task performance.

**The mechanics.** **Quantization** means storing numbers with fewer bits: a weight
normally kept in 16 or 32 bits is squeezed into 4 bits, roughly quartering the
base's memory footprint at the cost of some precision. QLoRA keeps that 4-bit base
**frozen** and trains a normal full-precision LoRA **adapter** on top, so gradients
flow only through the adapter. The quantized base loses a little accuracy, the
adapter makes it back up.

> CI honesty note: real QLoRA needs a true 4-bit GPU kernel (such as bitsandbytes),
> which is not available on a plain CPU CI runner. This lesson does NOT run real
> 4-bit math. It is a CPU concept demo: we *simulate* the precision loss with
> `fake_quantize`, which rounds the weights to 4-bit-like resolution while keeping
> them in normal floats, so the lesson runs anywhere and shows the *idea* without
> claiming the real memory savings.

In [ ]:
from pocketlm import (train_tiny, pick_config, fake_quantize, add_lora,
                      trainable_parameters, make_batch)

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
with torch.no_grad():
    for p in model.parameters():
        p.copy_(fake_quantize(p, bits=4))   # simulate the 4-bit frozen base
add_lora(model, rank=4)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)
opt = torch.optim.AdamW(trainable_parameters(model), lr=1e-3)
g = torch.Generator().manual_seed(0)
for _ in range(60 if os.environ.get("POCKETLM_CI") else 150):
    x, y = make_batch(data, model.cfg.block_size, 16, generator=g)
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
print("adapter trains over a 4-bit base, final loss:", round(loss.item(), 3))

**What just happened.** We rounded the frozen base toward 4-bit resolution, bolted
on a LoRA adapter, and trained only the adapter. The printed final loss is finite
and trains down: the adapter learned the task *despite* the coarsened base. On real
hardware the payoff would be a base that takes roughly a quarter of the memory,
which is what lets a large model fit on a small GPU. Remember the honesty note: here
the savings are simulated, the *behavior* (adapter recovering over a degraded base)
is what we are demonstrating, not real 4-bit kernels.

⚠️ **Common pitfalls**
- Believing this CPU demo gives the real memory savings. It does not. `fake_quantize`
  only mimics the precision loss, the floats are still full-width in RAM.
- Quantizing the adapter too. The whole point is a low-precision *base* with a
  full-precision adapter, so the adapter has the resolution to compensate.
- Assuming quantization is free. It always costs some accuracy, QLoRA is a trade
  that happens to land well, not a free lunch.

🏋️ **Try it yourself.** Lower the precision further, change `bits=4` to `bits=2`,
and rerun. Watch the final loss climb as the base gets coarser, then bump the
adapter `rank` up and see how much of that loss the adapter can claw back. You are
feeling the precision-versus-capacity balance at the heart of QLoRA.

In [ ]:
assert torch.isfinite(loss)